# Ridge and Lasso Regression

**Topic:** Supervised Learning — Regression

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Output, VBox, ToggleButtons, FloatLogSlider
from IPython.display import display, clear_output
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error
np.random.seed(42)
from tkh_utils import PALETTE, FONT, base_layout

In [2]:
_hous = fetch_california_housing(as_frame=True)
_N_SUB, _N_NOISE = 210, 60
_idx = np.random.RandomState(42).choice(len(_hous.target), _N_SUB, replace=False)
_Xreal = _hous.data.iloc[_idx].reset_index(drop=True).copy()
_y = _hous.target.values[_idx]

# 8 real features + 60 pure-noise columns on 210 rows. Deliberate: with the full
# 20k rows and only 8 informative features there is nothing to overfit, so
# regularization can only hurt and the widgets teach the opposite of the lesson.
_rng = np.random.RandomState(42)
for i in range(_N_NOISE):
    _Xreal[f"Noise_{i+1}"] = _rng.normal(0, 1, _N_SUB)

FEATURES = list(_Xreal.columns)
REAL_FEATURES = list(_hous.feature_names)          # first 8
NOISE_FEATURES = FEATURES[8:]
_Xs = StandardScaler().fit_transform(_Xreal)
X_train, X_test, y_train, y_test = train_test_split(
    _Xs, _y, test_size=0.25, random_state=42)

# Acceptance check: regularization must genuinely help here, not just shrink an
# already-fine fit. Requires real overfitting (train >> test R2) AND a genuine
# test-R2 win for the best regularized model over plain OLS.
_ols = LinearRegression().fit(X_train, y_train)
_ols_train_r2 = r2_score(y_train, _ols.predict(X_train))
_ols_test_r2 = r2_score(y_test, _ols.predict(X_test))

_ridge_alphas_check = np.logspace(-2, 4, 60)
_ridge_test_r2s_check = [r2_score(y_test, Ridge(alpha=a).fit(X_train, y_train).predict(X_test))
                          for a in _ridge_alphas_check]
_best_ridge_i = int(np.argmax(_ridge_test_r2s_check))

_lasso_alphas_check = np.logspace(-4, 0.5, 60)
_lasso_test_r2s_check = [r2_score(y_test, Lasso(alpha=a, max_iter=20000).fit(X_train, y_train).predict(X_test))
                          for a in _lasso_alphas_check]
_best_lasso_i = int(np.argmax(_lasso_test_r2s_check))

print(f"OLS         train R2={_ols_train_r2:.3f}  test R2={_ols_test_r2:.3f}  "
      f"gap={_ols_train_r2 - _ols_test_r2:.3f}")
print(f"Ridge best  test R2={_ridge_test_r2s_check[_best_ridge_i]:.3f} "
      f"@ alpha={_ridge_alphas_check[_best_ridge_i]:.4g}")
print(f"Lasso best  test R2={_lasso_test_r2s_check[_best_lasso_i]:.3f} "
      f"@ alpha={_lasso_alphas_check[_best_lasso_i]:.4g}")

assert (_ols_train_r2 - _ols_test_r2) >= 0.30, \
    "OLS overfitting gap too small — raise _N_NOISE or lower _N_SUB"
assert max(_ridge_test_r2s_check[_best_ridge_i], _lasso_test_r2s_check[_best_lasso_i]) - _ols_test_r2 >= 0.15, \
    "Best regularized model doesn't beat OLS by enough — raise _N_NOISE or lower _N_SUB"
print("Acceptance criterion met.")

OLS         train R2=0.810  test R2=0.189  gap=0.621
Ridge best  test R2=0.435 @ alpha=36.25
Lasso best  test R2=0.601 @ alpha=0.06638
Acceptance criterion met.


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** how Ridge and Lasso add a penalty term to the linear regression loss
- **Read** a coefficient path plot and explain why Lasso's lines reach zero and Ridge's never do
- **Explain** why a good alpha for Ridge tells you nothing about a good alpha for Lasso

> **Tip:** In **Coefficient Paths**, toggle between Ridge and Lasso and watch the gray band of 60 noise-feature lines. Under Lasso the band dies off by around alpha ≈ 0.2; under Ridge it never touches zero, no matter how far you drag the slider. In **What Regularization Buys You**, watch the test R² curve rise, peak, and fall as alpha increases — that peak is the best alpha, and cross-validation is how you find it without peeking at test data.

---
## How we got here

Regularization is the answer to the overfitting problem you saw in polynomial regression:

- **[supervised/03_polynomial_regression.ipynb](03_polynomial_regression.ipynb)** — the overfitting problem and why unconstrained MSE minimization goes wrong at high complexity
- **[math/statistics_for_ml/04_regularization_mathematics.ipynb](../math/statistics_for_ml/04_regularization_mathematics.ipynb)** — the mathematical derivation of why adding a penalty to the loss function shrinks coefficients
- **[ml_concepts/04_bias_variance_tradeoff.ipynb](../ml_concepts/04_bias_variance_tradeoff.ipynb)** — regularization is a deliberate way to increase bias slightly in exchange for a larger reduction in variance

---
## Why this matters for data science

Ridge and Lasso are the most widely used regularized linear models in practice. They solve two of the biggest problems with plain linear regression: **overfitting to noise and unstable coefficients when features are correlated.**

Lasso has a unique and extremely valuable property: it performs **automatic feature selection** by setting irrelevant coefficients to exactly zero. When you have hundreds of candidate features and only some of them matter, Lasso finds the sparse subset for you. Ridge is better when all features are relevant and you just want to reduce their magnitudes. Elastic Net combines both.

---
## Where it sits on the spectrum

See **[ml_concepts/13_interpretability_vs_complexity.ipynb](../ml_concepts/13_interpretability_vs_complexity.ipynb)** for the full spectrum.

Ridge and Lasso sit at the **same top-left position** as linear regression, but they are more robust to high-dimensional or correlated feature spaces. The penalty shrinks the model's effective complexity even when you have many features, keeping interpretability intact.

As alpha increases (stronger regularization), the model moves further down and to the left: it becomes more constrained, simpler, and more biased — but also less prone to variance. Push alpha far enough and nearly every coefficient collapses toward zero: the model stops being a rich combination of features and becomes close to predicting the same constant value for every input.

---
## How it learns

Think about what happens when you fit linear regression on noisy data with many features. The algorithm will happily assign a large coefficient to a feature that by chance correlates with the noise in the training set. That large coefficient is not real — it is just overfitting.

Regularization fixes this by taxing large coefficients. The model now minimizes not just prediction error but also the size of its own weights. To earn a large coefficient, a feature has to produce a big enough reduction in MSE to pay the penalty. Features that only correlate with noise do not make that trade — their coefficients shrink or disappear.

Ridge and Lasso use different penalties. Ridge charges proportional to the **squared** size of each coefficient. In practice this shrinks every coefficient toward zero without ever arriving there, and it leans hardest on features that duplicate information already captured elsewhere — two correlated features get shrunk far more aggressively than two independent ones carrying the same total signal. Lasso charges proportional to the **absolute** size — it has a corner in the cost function geometry that lets a coefficient land exactly on zero, not just close to it.

---
## The math behind it

Both methods add a **regularization term** to the squared-error loss — but not the same one, and not on the same scale as each other. This matters because it's exactly what sklearn implements.

**Ridge (L2 regularization)** minimizes:

$$J_{\text{Ridge}}(\boldsymbol{\theta}) = \sum_{i=1}^{n}(\hat{y}_i - y_i)^2 + \alpha \sum_{j=1}^{p} \theta_j^2$$

**Lasso (L1 regularization)** minimizes:

$$J_{\text{Lasso}}(\boldsymbol{\theta}) = \frac{1}{2n}\sum_{i=1}^{n}(\hat{y}_i - y_i)^2 + \alpha \sum_{j=1}^{p} |\theta_j|$$

- $\alpha$ — regularization strength (hyperparameter); $\alpha = 0$ gives plain linear regression
- The penalty applies only to feature coefficients $\theta_1, \ldots, \theta_p$, not the intercept $\theta_0$

Notice Ridge's error term is **not** divided by the number of rows $n$, while Lasso's is divided by $2n$. These two alphas are not on the same scale. That's not a deep fact about L1 versus L2 — it's that sklearn's `Ridge` doesn't divide its error term by the number of rows and sklearn's `Lasso` does. The practical consequence: a good Ridge alpha depends on how much data you have and must be retuned when your dataset grows. A good Lasso alpha doesn't.

**Why Lasso produces zeros:** the L1 penalty has a non-differentiable point at $\theta_j = 0$. The subdifferential of the absolute value function includes zero, so the optimization can land exactly at zero and stay there. Ridge uses a smooth quadratic penalty that approaches but never reaches zero.

**Optimization criterion:** minimize the penalized loss above. Ridge has a closed-form solution; Lasso uses coordinate descent because the L1 penalty is not differentiable at zero. The intercept $\theta_0$ is never penalized in either model.

---
## Try it yourself

Two widgets below: **Coefficient Paths** shows how all 68 coefficients move as regularization strength increases, and **What Regularization Buys You** shows what that shrinkage costs you in training fit and buys you back in generalization.

### Coefficient Paths

In [ ]:
# Precompute coefficient paths once — the render loop below only looks up
# cached values, it never refits.
_ridge_alphas_w1 = np.logspace(-2, 4, 60)
_lasso_alphas_w1 = np.logspace(-4, 0.5, 60)

def _coef_path(model_cls, alphas, **kwargs):
    coefs = np.zeros((len(alphas), len(FEATURES)))
    for i, a in enumerate(alphas):
        coefs[i] = model_cls(alpha=a, **kwargs).fit(X_train, y_train).coef_
    return coefs

_PATHS_W1 = {
    "Ridge": (_ridge_alphas_w1, _coef_path(Ridge, _ridge_alphas_w1)),
    "Lasso": (_lasso_alphas_w1, _coef_path(Lasso, _lasso_alphas_w1, max_iter=20000)),
}
_SLIDER_RANGE_W1 = {"Ridge": (-2, 4), "Lasso": (-4, 0.5)}

# 3 colors from PALETTE plus 5 extensions - 8 real features need 8 distinct,
# legend-worthy colors, more than PALETTE's 3 non-muted entries provide.
_REAL_COLORS_W1 = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"],
                   "#AE3EC9", "#1098AD", "#E64980", "#F59F00", "#495057"]

out_path = Output()
model_tg_w1 = ToggleButtons(options=["Ridge", "Lasso"], value="Ridge")
alpha_sl_w1 = FloatLogSlider(value=1.0, base=10, min=-2, max=4, step=0.1017,
    description="alpha:", style={"description_width": "60px"},
    layout=widgets.Layout(width="420px"))
cap_w1 = widgets.HTML()

_initialized_w1 = False

def render_path(change=None):
    if not _initialized_w1:
        return
    model_name = model_tg_w1.value
    alphas, coefs = _PATHS_W1[model_name]
    idx = int(np.argmin(np.abs(np.log10(alphas) - np.log10(alpha_sl_w1.value))))
    alpha = alphas[idx]
    coef_row = coefs[idx]

    fig = go.Figure()
    for j, feat in enumerate(NOISE_FEATURES):
        fig.add_trace(go.Scatter(
            x=alphas, y=coefs[:, 8 + j], mode="lines",
            line=dict(color=PALETTE["muted"], width=1),
            opacity=0.35, name="60 noise features",
            legendgroup="noise", showlegend=(j == 0), hoverinfo="skip",
        ))
    for j, feat in enumerate(REAL_FEATURES):
        fig.add_trace(go.Scatter(
            x=alphas, y=coefs[:, j], mode="lines",
            line=dict(color=_REAL_COLORS_W1[j], width=2.5), name=feat,
        ))
    fig.update_layout(base_layout(
        title=f"{model_name} coefficient paths",
        xaxis_title="alpha (log scale)",
        yaxis_title="Coefficient value",
    ))
    fig.update_xaxes(type="log")
    fig.add_vline(x=alpha, line_dash="dash", line_color="black")
    fig.add_hline(y=0, line_color="black", line_width=1)

    with out_path:
        clear_output(wait=True)
        fig.show()

    n_real_alive = int(np.sum(~np.isclose(coef_row[:8], 0.0, atol=1e-6)))
    n_noise_alive = int(np.sum(~np.isclose(coef_row[8:], 0.0, atol=1e-6)))
    cap_w1.value = (f"<b>{model_name} at α={alpha:.4g}:</b> {n_real_alive} of 8 real features "
                     f"still alive, {n_noise_alive} of 60 noise features still alive.")

def _on_model_change_w1(change=None):
    new_min, new_max = _SLIDER_RANGE_W1[model_tg_w1.value]
    new_step = (new_max - new_min) / 59
    new_value = 10 ** ((new_min + new_max) / 2)
    alpha_sl_w1.min = min(alpha_sl_w1.min, new_min)
    alpha_sl_w1.max = max(alpha_sl_w1.max, new_max)
    alpha_sl_w1.value = new_value
    alpha_sl_w1.step = new_step
    alpha_sl_w1.min = new_min
    alpha_sl_w1.max = new_max
    render_path()

model_tg_w1.observe(_on_model_change_w1, names="value")
alpha_sl_w1.observe(render_path, names="value")

display(VBox([model_tg_w1, alpha_sl_w1, out_path, cap_w1]))
_initialized_w1 = True
render_path()

Drag the slider right and watch the gray band. Under **Lasso**, noise features hit exactly zero and stay there: by α ≈ 0.19 all 60 are gone. Under **Ridge**, no line ever touches zero, no matter how far right you drag: the smallest coefficient is still around 0.00004 at α=10,000 — shrunk, but never exactly zero. That's the whole difference between the two, and it's why Lasso is the only linear model that does feature selection for you.

One sentence of theory: Lasso's penalty has a sharp corner at zero and Ridge's is smooth, and a corner is something a solution can land exactly on.

### What Regularization Buys You

In [ ]:
_ridge_alphas_w2 = np.logspace(-2, 4, 60)
_lasso_alphas_w2 = np.logspace(-4, 0.5, 60)

def _r2_curve(model_cls, alphas, **kwargs):
    train_r2 = np.zeros(len(alphas))
    test_r2 = np.zeros(len(alphas))
    for i, a in enumerate(alphas):
        m = model_cls(alpha=a, **kwargs).fit(X_train, y_train)
        train_r2[i] = r2_score(y_train, m.predict(X_train))
        test_r2[i] = r2_score(y_test, m.predict(X_test))
    return train_r2, test_r2

_ridge_train_r2_w2, _ridge_test_r2_w2 = _r2_curve(Ridge, _ridge_alphas_w2)
_lasso_train_r2_w2, _lasso_test_r2_w2 = _r2_curve(Lasso, _lasso_alphas_w2, max_iter=20000)

_CURVES_W2 = {
    "Ridge": (_ridge_alphas_w2, _ridge_train_r2_w2, _ridge_test_r2_w2),
    "Lasso": (_lasso_alphas_w2, _lasso_train_r2_w2, _lasso_test_r2_w2),
}
_SLIDER_RANGE_W2 = {"Ridge": (-2, 4), "Lasso": (-4, 0.5)}

_ols_w2 = LinearRegression().fit(X_train, y_train)
_ols_train_r2_w2 = r2_score(y_train, _ols_w2.predict(X_train))
_ols_test_r2_w2 = r2_score(y_test, _ols_w2.predict(X_test))

out_buys = Output()
model_tg_w2 = ToggleButtons(options=["Ridge", "Lasso"], value="Ridge")
alpha_sl_w2 = FloatLogSlider(value=1.0, base=10, min=-2, max=4, step=0.1017,
    description="alpha:", style={"description_width": "60px"},
    layout=widgets.Layout(width="420px"))
cap_w2 = widgets.HTML()

_initialized_w2 = False

def render_buys(change=None):
    if not _initialized_w2:
        return
    model_name = model_tg_w2.value
    alphas, train_r2, test_r2 = _CURVES_W2[model_name]
    idx = int(np.argmin(np.abs(np.log10(alphas) - np.log10(alpha_sl_w2.value))))
    alpha = alphas[idx]
    best_idx = int(np.argmax(test_r2))

    fig = go.Figure(data=[
        go.Scatter(x=alphas, y=train_r2, mode="lines", name="Train R²",
                   line=dict(color=PALETTE["primary"], width=2.5)),
        go.Scatter(x=alphas, y=test_r2, mode="lines", name="Test R²",
                   line=dict(color=PALETTE["secondary"], width=2.5)),
        go.Scatter(x=[alphas[best_idx]], y=[test_r2[best_idx]], mode="markers",
                   marker=dict(symbol="star", size=14, color=PALETTE["accent"]),
                   name="Best test R²"),
    ], layout=base_layout(
        title=f"{model_name}: train vs. test R² across alpha",
        xaxis_title="alpha (log scale)",
        yaxis_title="R²",
    ))
    fig.update_xaxes(type="log")
    fig.add_vline(x=alpha, line_dash="dash", line_color="black")
    fig.add_hline(y=_ols_test_r2_w2, line_dash="dot", line_color=PALETTE["muted"],
                  annotation_text="no regularization", annotation_position="bottom right")

    with out_buys:
        clear_output(wait=True)
        fig.show()

    cap_w2.value = (
        f"<b>{model_name} at α={alpha:.4g}:</b> train R² {train_r2[idx]:.2f}, "
        f"test R² {test_r2[idx]:.2f}. Without regularization: train R² {_ols_train_r2_w2:.2f}, "
        f"test R² {_ols_test_r2_w2:.2f}. You gave up training fit and bought back generalization."
    )

def _on_model_change_w2(change=None):
    new_min, new_max = _SLIDER_RANGE_W2[model_tg_w2.value]
    new_step = (new_max - new_min) / 59
    new_value = 10 ** ((new_min + new_max) / 2)
    alpha_sl_w2.min = min(alpha_sl_w2.min, new_min)
    alpha_sl_w2.max = max(alpha_sl_w2.max, new_max)
    alpha_sl_w2.value = new_value
    alpha_sl_w2.step = new_step
    alpha_sl_w2.min = new_min
    alpha_sl_w2.max = new_max
    render_buys()

model_tg_w2.observe(_on_model_change_w2, names="value")
alpha_sl_w2.observe(render_buys, names="value")

display(VBox([model_tg_w2, alpha_sl_w2, out_buys, cap_w2]))
_initialized_w2 = True
render_buys()

Slide alpha from left to right. On the left, training R² is near-perfect and test R² is poor: the model has memorized 157 training rows including 60 columns of pure noise. As alpha rises, training R² falls and test R² **climbs**. That crossover is the entire point of regularization. Push alpha too far right and both collapse: that's underfitting. The best alpha is the peak of the test curve, and cross-validation is how you find it without peeking at the test set.

---
## What's happening?

In **Coefficient Paths**, alpha controls how heavy the tax on large coefficients is. Lasso reaches full sparsity (every noise feature zeroed) by around alpha ≈ 0.19, while Ridge is still shrinking, not eliminating, even at alpha = 10,000. That's the durable, real part of the difference between the two penalties: **L1 reaches exact zero at a finite alpha; L2 only approaches zero asymptotically.**

The alpha *ranges* the two sliders use, though, are mostly an artifact of how sklearn defines the two losses (see **The math behind it** above): Ridge's loss isn't divided by the number of training rows and Lasso's is, so the two alphas live on different scales and neither tells you anything about the other. You can see the scale-with-n effect directly: on this notebook's 157-row training set, Ridge's best test-R² alpha is about 36. Fit the same 8 real features plus 60 noise features on the full ~15,480-row training set instead, and the best alpha rises to about 187 — more than 5 times larger, for the identical penalty and features, just because n changed.

In **What Regularization Buys You**, the curve shows both train and test R² responding to alpha. As alpha increases, train R² falls steadily — the model gives up fitting the training data exactly — while test R² rises to a peak and then falls again. The best alpha is that peak, and cross-validation is how you find it without ever looking at the test set.

---
## Key hyperparameters

**`alpha`** (default `1.0`) — regularization strength. Larger values shrink coefficients more aggressively. This is the single most important hyperparameter. Always tune it with cross-validation (RidgeCV or LassoCV). Remember: Ridge's alpha and Lasso's alpha are not on the same scale — see the normalization note in **The math behind it** above.

**`max_iter`** (default `1000` for Lasso) — maximum number of coordinate descent iterations. Increase this if Lasso does not converge on your dataset. Ridge does not need this (it has a closed-form solution).

**`tol`** (default `1e-4`) — convergence tolerance for iterative solvers (Lasso). Lower values give more precise solutions but take longer.

For the full list of hyperparameters, see the sklearn documentation:
[Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) |
[Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)

---
## Strengths and weaknesses

| Strengths | Weaknesses |
|-----------|------------|
| Solves multicollinearity and overfitting in one step | Adds a hyperparameter (alpha) that must be tuned |
| Lasso does automatic feature selection (exact zeros) | Lasso can arbitrarily pick one of several correlated features and zero out the rest |
| Ridge is numerically stable even when p > n | Very high alpha forces underfitting |
| Fast to train and interpretable (still linear coefficients) | Standard errors and p-values require extra work (use statsmodels for inference) |
| Elastic Net combines both penalties for the best of both | Scale-sensitive: features must be standardized before regularization means anything |

---
## When to use it / When NOT to use it

| Use it when | Do NOT use it when |
|---|---|
| You have many features and suspect some are irrelevant (Lasso) | You need individual coefficient significance (use plain linear regression + statsmodels) |
| Features are correlated and plain linear regression is unstable (Ridge) | Your features are already sparse or you know all are important (regularization adds unnecessary bias) |
| You have more features than training examples (p > n) | You are using count or text data with nonnegative constraints (a nonnegative *linear* model may still underfit — consider Poisson/GLM models) |
| You need nonnegative coefficients for interpretability (both `Ridge(positive=True)` and `Lasso(positive=True)` support this) | The nonlinearity in the data is large (regularized linear still underfits) |
| You are fitting polynomial features and worry about overfitting | You need online/incremental learning (use SGDRegressor with L1 or L2 penalty) |

In [5]:
alphas_ridge_cv = np.logspace(-2, 5, 200)
alphas_lasso_cv = np.logspace(-4, 1, 200)
ridge_cv = RidgeCV(alphas=alphas_ridge_cv, cv=5).fit(X_train, y_train)
lasso_cv = LassoCV(alphas=alphas_lasso_cv, cv=5, max_iter=20000, random_state=42).fit(X_train, y_train)

for name, model in [("Ridge", ridge_cv), ("Lasso", lasso_cv)]:
    y_pred = model.predict(X_test)
    nonzero = int(np.sum(~np.isclose(model.coef_, 0.0, atol=1e-6)))
    print(f"{name:6s}  alpha={model.alpha_:.4g}  "
          f"R²={r2_score(y_test, y_pred):.4f}  "
          f"RMSE={root_mean_squared_error(y_test, y_pred):.4f}  "
          f"nonzero_coefs={nonzero}/{len(FEATURES)}")

zeroed_by_lasso = [f for f, c in zip(REAL_FEATURES, lasso_cv.coef_[:8]) if np.isclose(c, 0.0, atol=1e-6)]
ridge_noise_alive = int(np.sum(~np.isclose(ridge_cv.coef_[8:], 0.0, atol=1e-6)))
lasso_noise_alive = int(np.sum(~np.isclose(lasso_cv.coef_[8:], 0.0, atol=1e-6)))
print(f"\nReal features zeroed by Lasso: {zeroed_by_lasso}")
print(f"Noise features still alive — Ridge: {ridge_noise_alive}/60   Lasso: {lasso_noise_alive}/60")

fig = make_subplots(rows=1, cols=2, column_widths=[0.7, 0.3],
                     subplot_titles=("8 real features", "60 noise features surviving"))
fig.add_trace(go.Bar(name=f"Ridge (α={ridge_cv.alpha_:.3g})", x=REAL_FEATURES, y=ridge_cv.coef_[:8],
                      marker_color=PALETTE["primary"], opacity=0.85), row=1, col=1)
fig.add_trace(go.Bar(name=f"Lasso (α={lasso_cv.alpha_:.4g})", x=REAL_FEATURES, y=lasso_cv.coef_[:8],
                      marker_color=PALETTE["secondary"], opacity=0.85), row=1, col=1)
fig.add_trace(go.Bar(x=["Ridge", "Lasso"], y=[ridge_noise_alive, lasso_noise_alive],
                      marker_color=[PALETTE["primary"], PALETTE["secondary"]], showlegend=False),
              row=1, col=2)
fig.update_layout(base_layout(
    title="Ridge vs Lasso — cross-validated coefficients (8 real features) and noise survival",
    xaxis_title="", yaxis_title="Coefficient",
))
fig.update_layout(barmode="group")
fig.update_xaxes(tickangle=-45, row=1, col=1)
fig.update_yaxes(title_text="# noise features nonzero", row=1, col=2)
fig.show()

Ridge   alpha=7.067  R²=0.3484  RMSE=0.8526  nonzero_coefs=68/68
Lasso   alpha=0.03872  R²=0.5774  RMSE=0.6866  nonzero_coefs=33/68

Real features zeroed by Lasso: ['AveBedrms']
Noise features still alive — Ridge: 60/60   Lasso: 26/60


---
## Real-world example: Ridge vs. Lasso, cross-validated

The chart above uses the same 8 real California Housing features plus 60 synthetic noise features from the shared dataset built at the top of this notebook. Ridge and Lasso are each tuned by 5-fold cross-validation on the training set, and the difference in what they do with the noise features is the whole point of this example.

- **Notice:** Ridge (α≈7.07) keeps all 68 features nonzero, including all 60 noise columns — it down-weights weak signals but never eliminates them
- **Notice:** Lasso (α≈0.039) zeros out 34 of the 60 noise features, plus `AveBedrms` — a real feature, but one that's highly correlated with `AveRooms`
- **Notice:** Lasso achieves meaningfully better test R² here (0.577 vs. 0.348 for Ridge), using roughly half the features — a real benefit for interpretability and deployment, not just a theoretical one

> **Discussion question:** In a medical risk model, would you prefer Ridge or Lasso? Does the answer change if doctors need to understand why the model made a specific prediction?

### Regularization methods comparison

| Method | Penalty | Feature selection | Best for |
|---|---|---|---|
| Ridge (L2) | Sum of squared coefs | No: all features kept | Correlated features, stable coefficients |
| Lasso (L1) | Sum of absolute coefs | Yes: exact zeros | Many irrelevant features, sparse models |
| Elastic Net | Ridge + Lasso combined | Partial | Both correlated features and irrelevant ones |
| Plain Linear | None | No | When you want maximum statistical power |

> **Ridge shrinks all coefficients toward zero proportionally; Lasso drives some to exactly zero, making it the only linear model that performs automatic feature selection.**

---
*Next up: 05 — Logistic Regression, the classification counterpart that turns a linear score into a probability*